# Complete reproducible GPU workflow (Kaggle or Colab)

This notebook consolidates asset preparation, embeddings, supervised selector training, resumable PPO, final evaluation, and inference export. Expensive stages are controlled by explicit flags. On Kaggle, attach the private `cs-769-ppo-assets` dataset, enable a GPU and internet, and use **Save Version -> Save & Run All** so `/kaggle/working/cs769_outputs` becomes a persistent notebook output. The final cell retains the completed experiment summary so results remain visible when the notebook is opened.

In [ ]:
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
WORK_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path('/content')
PROJECT_ROOT = WORK_ROOT / 'CS_769_Project'

if not PROJECT_ROOT.exists():
    !git clone https://github.com/Achintya2510/CS_769_Project.git {PROJECT_ROOT}

%cd {PROJECT_ROOT}
!git pull --ff-only origin main
requirements_path = PROJECT_ROOT / 'requirements.txt'
%pip install -q -r {requirements_path}

print('Platform:', 'Kaggle' if IS_KAGGLE else 'Colab')
print('Project:', PROJECT_ROOT)

In [ ]:
import torch
import transformers
import sentence_transformers

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Sentence Transformers:', sentence_transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a GPU accelerator before continuing.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import importlib
import json
import shutil
import subprocess
from pathlib import Path

import torch
import training

importlib.reload(training)
from data_pipeline import load_config, load_jsonl
from models import (FixedAnswerer, FrozenTextEncoder, build_actor_critic,
                    create_embedding_cache, load_embedding_cache)

config = load_config(PROJECT_ROOT / 'config.yaml')

if IS_KAGGLE:
    input_root = Path('/kaggle/input')
    processed_archives = list(input_root.rglob('processed.tar.gz'))
    outputs_archives = list(input_root.rglob('outputs.tar.gz'))
    if not processed_archives or not outputs_archives:
        raise FileNotFoundError(
            'Attach the private cs-769-ppo-assets Kaggle dataset first.'
        )

    runtime_root = Path('/tmp/cs769_runtime')
    processed_dir = runtime_root / 'processed'
    output_dir = Path('/kaggle/working/cs769_outputs')
    runtime_root.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not processed_dir.exists():
        print('Extracting processed data...')
        subprocess.run(
            ['tar', '-xzf', str(processed_archives[0]), '-C', str(runtime_root)],
            check=True,
        )

    # A previous Kaggle notebook output takes priority over the original upload.
    previous_outputs = [
        path for path in input_root.rglob('cs769_outputs')
        if (path / 'colab_runs').exists()
    ]
    if previous_outputs:
        print('Restoring previous Kaggle output:', previous_outputs[0])
        shutil.copytree(previous_outputs[0], output_dir, dirs_exist_ok=True)
    elif not (output_dir / 'embeddings').exists():
        print('Extracting embeddings and checkpoints...')
        subprocess.run(
            ['tar', '-xzf', str(outputs_archives[0]), '-C', str(output_dir)],
            check=True,
        )
else:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive/CS_769_Project')
    processed_dir = drive_root / 'data' / 'processed'
    output_dir = drive_root / 'outputs'

config['paths']['processed_dir'] = str(processed_dir)
config['paths']['output_dir'] = str(output_dir)
embedding_dir = output_dir / 'embeddings'
device = torch.device('cuda')

# Loading table_index.json once avoids thousands of repeated reads.
table_index = json.loads((processed_dir / 'table_index.json').read_text())
def fast_load_processed_table(root, table_id):
    root = Path(root)
    return json.loads(
        (root / 'tables' / table_index[str(table_id)]).read_text()
    )
training.load_processed_table = fast_load_processed_table

from training import (
    build_environment_factory,
    export_inference_bundle,
    mean_greedy_episode_reward,
    mean_greedy_evidence_f1,
    set_seed,
    train_ppo,
    train_supervised,
)

print('Processed data:', processed_dir)
print('Persistent output:', output_dir)
print('PPO minibatch size:', config['training']['ppo_minibatch_size'])

In [ ]:
runs_dir = output_dir / 'colab_runs'
required_files = [
    embedding_dir / 'evidence_embeddings.npy',
    embedding_dir / 'evidence_index.json',
    embedding_dir / 'question_embeddings.npy',
    embedding_dir / 'question_index.json',
]
for seed in config['training']['seeds']:
    required_files.append(runs_dir / f'supervised_seed_{seed}' / 'best_model.pt')

missing = [path for path in required_files if not path.exists()]
if missing:
    print('Fresh setup is incomplete. Run the embedding and supervised cells below before PPO:')
    print(*missing, sep='\n')

else:
    print('All embeddings and supervised models found.')
checkpoints = sorted(runs_dir.glob('ppo_*/latest.pt'))
if not checkpoints:
    print('No PPO checkpoint found; training would begin at episode 0.')
for checkpoint_path in checkpoints:
    state = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    progress = state['progress']
    print(
        checkpoint_path.parent.name,
        f"episode {progress['episode']}/{config['training']['max_episodes']}",
        f"checks_without_improvement={progress.get('checks_without_improvement', 0)}",
    )
    del state
print('Answer-reward cache:', (output_dir / 'answer_reward_cache.json').exists())

## PPO resume

Run one variant and one seed per Kaggle version. The matching `latest.pt` restores the model, optimizer, RNG state, logs and episode. The first continuation target is `answer_only`, seed `42`.

In [ ]:
RUN_PPO = False
TARGET_VARIANT = 'answer_only'
TARGET_SEED = 42
VALID_VARIANTS = {'answer_only', 'evidence_only', 'combined', 'combined_step'}

if RUN_PPO:
    if TARGET_VARIANT not in VALID_VARIANTS:
        raise ValueError(f'Unknown PPO variant: {TARGET_VARIANT}')
    if TARGET_SEED not in config['training']['seeds']:
        raise ValueError(f'Unknown training seed: {TARGET_SEED}')

    run_dir = output_dir / 'colab_runs' / f'ppo_{TARGET_VARIANT}_seed_{TARGET_SEED}'
    latest_path = run_dir / 'latest.pt'
    should_train = True
    if latest_path.exists():
        saved = torch.load(latest_path, map_location='cpu', weights_only=False)
        progress = saved['progress']
        episode = int(progress.get('episode', 0))
        stale_checks = int(progress.get('checks_without_improvement', 0))
        patience = int(config['training'].get('ppo_early_stopping_patience', 0))
        should_train = not (
            episode >= int(config['training']['max_episodes'])
            or (patience > 0 and stale_checks >= patience)
        )
        print(f'Resumable checkpoint: episode {episode}')
        del saved

    if not should_train:
        print('This run is already complete or early-stopped; nothing was retrained.')
    else:
        from evaluation import AnswerRewardCache

        evidence_embeddings, question_embeddings = load_embedding_cache(embedding_dir)
        train_rows = load_jsonl(processed_dir / 'questions' / 'train_core.jsonl')
        val_rows = load_jsonl(processed_dir / 'questions' / 'train_val.jsonl')
        question_ids = [row['question_id'] for row in train_rows]
        reward_callback = None

        if TARGET_VARIANT != 'evidence_only':
            examples_by_id = {row['question_id']: row for row in train_rows + val_rows}
            _, train_evidence = build_environment_factory(
                train_rows, processed_dir, evidence_embeddings, question_embeddings,
                config, reward_variant='evidence_only',
            )
            _, val_evidence = build_environment_factory(
                val_rows, processed_dir, evidence_embeddings, question_embeddings,
                config, reward_variant='evidence_only',
            )
            answerer = FixedAnswerer(
                config['models']['answer_model'],
                config['models']['answer_model_revision'],
                device='cuda',
                max_input_tokens=config['models']['max_input_tokens'],
                max_output_tokens=config['models']['max_output_tokens'],
            )
            reward_callback = AnswerRewardCache(
                output_dir / 'answer_reward_cache.json',
                answerer,
                examples_by_id,
                {**train_evidence, **val_evidence},
                config['models']['answer_model'],
            )

        env_factory, _ = build_environment_factory(
            train_rows, processed_dir, evidence_embeddings, question_embeddings,
            config, answer_reward_fn=reward_callback, reward_variant=TARGET_VARIANT,
        )
        val_factory, _ = build_environment_factory(
            val_rows, processed_dir, evidence_embeddings, question_embeddings,
            config, answer_reward_fn=reward_callback, reward_variant=TARGET_VARIANT,
        )

        set_seed(TARGET_SEED)
        model = build_actor_critic(config).to(device)
        supervised_path = (
            output_dir / 'colab_runs' / f'supervised_seed_{TARGET_SEED}' / 'best_model.pt'
        )
        model.load_state_dict(
            torch.load(supervised_path, map_location=device, weights_only=True)
        )
        validation_ids = [row['question_id'] for row in val_rows[:500]]
        train_ppo(
            model, env_factory, question_ids, config, run_dir, device,
            validation_fn=lambda current: mean_greedy_episode_reward(
                current, val_factory, validation_ids, device
            ),
        )
        print('Finished:', run_dir)

### Preserve the Kaggle checkpoint

Kaggle writes checkpoints to `/kaggle/working/cs769_outputs`. Use **Save Version -> Save & Run All** so that directory is retained as notebook output. Before the next variant/seed, attach the preceding notebook output as an input; the setup cell automatically restores its `cs769_outputs` directory before training.

## Final evaluation and export (leave disabled until every PPO run is complete)

In [ ]:
RUN_FINAL_EVALUATION = False
RUN_EXPORT = False
FINAL_VARIANT = 'answer_only'
FINAL_SEED = 42
EVAL_VARIANTS = ['answer_only', 'evidence_only', 'combined', 'combined_step']

if RUN_FINAL_EVALUATION:
    from evaluation import (
        generate_answers, prepare_direct_inputs, prepare_policy_inputs,
        prepare_similarity_inputs, save_jsonl,
    )
    evidence_embeddings, question_embeddings = load_embedding_cache(embedding_dir)
    dev_rows = load_jsonl(processed_dir / 'questions' / 'dev.jsonl')
    dev_factory, evidence_by_id = build_environment_factory(
        dev_rows, processed_dir, evidence_embeddings, question_embeddings,
        config, reward_variant='evidence_only',
    )
    answerer = FixedAnswerer(
        config['models']['answer_model'], config['models']['answer_model_revision'],
        device='cuda', max_input_tokens=config['models']['max_input_tokens'],
        max_output_tokens=config['models']['max_output_tokens'],
    )
    result_dir = output_dir / 'results'
    baseline_inputs = {
        'direct': prepare_direct_inputs(dev_rows, processed_dir),
        'similarity': prepare_similarity_inputs(dev_rows, dev_factory, evidence_by_id),
    }
    for name, inputs in baseline_inputs.items():
        save_jsonl(result_dir / f'{name}.jsonl', generate_answers(answerer, inputs, batch_size=16))
    run_specs = [('supervised', None, seed) for seed in config['training']['seeds']]
    run_specs += [
        ('ppo', variant, seed)
        for variant in EVAL_VARIANTS for seed in config['training']['seeds']
    ]
    for kind, variant, seed in run_specs:
        model = build_actor_critic(config).to(device)
        if kind == 'supervised':
            path = output_dir / 'colab_runs' / f'supervised_seed_{seed}' / 'best_model.pt'
            state = torch.load(path, map_location=device, weights_only=True)
            name = f'supervised_seed_{seed}'
        else:
            path = output_dir / 'colab_runs' / f'ppo_{variant}_seed_{seed}' / 'best.pt'
            state = torch.load(path, map_location=device, weights_only=False)['model']
            name = f'ppo_{variant}_seed_{seed}'
        model.load_state_dict(state)
        inputs = prepare_policy_inputs(model, dev_rows, dev_factory, evidence_by_id, device)
        save_jsonl(result_dir / f'{name}.jsonl', generate_answers(answerer, inputs, batch_size=16))
    print(result_dir)

if RUN_EXPORT:
    import numpy as np
    final_model = build_actor_critic(config).to(device)
    checkpoint_path = (
        output_dir / 'colab_runs' / f'ppo_{FINAL_VARIANT}_seed_{FINAL_SEED}' / 'best.pt'
    )
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    final_model.load_state_dict(checkpoint['model'])
    evidence_matrix = np.load(embedding_dir / 'evidence_embeddings.npy')
    evidence_index = json.loads((embedding_dir / 'evidence_index.json').read_text())
    preprocessing_manifest = json.loads(
        (processed_dir / 'preprocessing_manifest.json').read_text()
    )
    bundle_dir = output_dir / 'final_bundle'
    manifest = export_inference_bundle(
        bundle_dir, final_model, config, evidence_matrix,
        evidence_index, preprocessing_manifest,
    )
    print(json.dumps(manifest, indent=2)[:3000])

## Fresh setup stages: embeddings and supervised initialization

These cells are included for full reproducibility. Run them before PPO only when the attached assets do not already contain embeddings and the three supervised checkpoints. They are disabled by default.

In [ ]:
RUN_EMBEDDINGS = False
if RUN_EMBEDDINGS:
    embedding_dir = output_dir / 'embeddings'
    encoder = FrozenTextEncoder(
        config['models']['evidence_encoder'],
        config['models']['evidence_encoder_revision'],
        device='cuda',
    )
    counts = create_embedding_cache(
        encoder, processed_dir, embedding_dir,
        batch_size=config['training']['supervised_batch_size'],
    )
    print('Embedding cache created:', counts)
else:
    print('Embedding stage skipped; existing cache will be reused.')

In [ ]:
RUN_SUPERVISED = False
if RUN_SUPERVISED:
    evidence_embeddings, question_embeddings = load_embedding_cache(embedding_dir)
    train_rows = load_jsonl(processed_dir / 'questions' / 'train_core.jsonl')
    val_rows = load_jsonl(processed_dir / 'questions' / 'train_val.jsonl')
    train_ids = [row['question_id'] for row in train_rows]
    validation_ids = [row['question_id'] for row in val_rows[:config['training']['validation_examples']]]
    train_factory, _ = build_environment_factory(
        train_rows, processed_dir, evidence_embeddings, question_embeddings,
        config, reward_variant='evidence_only',
    )
    val_factory, _ = build_environment_factory(
        val_rows, processed_dir, evidence_embeddings, question_embeddings,
        config, reward_variant='evidence_only',
    )
    for seed in config['training']['seeds']:
        run_dir = output_dir / 'colab_runs' / f'supervised_seed_{seed}'
        if (run_dir / 'best_model.pt').exists():
            print('Skipping existing checkpoint:', run_dir.name)
            continue
        set_seed(seed)
        model = build_actor_critic(config).to(device)
        history = train_supervised(
            model, train_factory, train_ids,
            epochs=config['training']['supervised_epochs'],
            learning_rate=config['training']['supervised_learning_rate'],
            device=device,
            validation_fn=lambda current: mean_greedy_evidence_f1(
                current, val_factory, validation_ids, device
            ),
            batch_size=config['training']['supervised_batch_size'],
            output_dir=run_dir,
        )
        print(f'Completed supervised seed {seed}: {len(history)} epochs')
else:
    print('Supervised stage skipped; existing best_model.pt files will be reused.')

## Optional one-time Colab-to-Kaggle asset publication

Run this only in Colab after preprocessing, embeddings, and supervised training are complete. It packages the persistent Drive artifacts into the private Kaggle dataset consumed by the setup cells at the top of this notebook.

In [ ]:
RUN_ASSET_PUBLISH = False
if RUN_ASSET_PUBLISH:
    if IS_KAGGLE:
        raise RuntimeError('Publish assets from Colab, not from Kaggle.')
    import subprocess
    import sys
    from google.colab import files

    uploaded = files.upload()  # Select kaggle.json.
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('Upload kaggle.json.')
    credential_dir = Path('/root/.kaggle')
    credential_dir.mkdir(parents=True, exist_ok=True)
    credential = credential_dir / 'kaggle.json'
    credential.write_bytes(uploaded['kaggle.json'])
    credential.chmod(0o600)
    asset_root = Path('/content/cs769_kaggle_assets')
    asset_root.mkdir(parents=True, exist_ok=True)
    drive_project = Path('/content/drive/MyDrive/CS_769_Project')
    subprocess.run(['tar', '-czf', str(asset_root / 'processed.tar.gz'),
                    '-C', str(drive_project / 'data'), 'processed'], check=True)
    subprocess.run(['tar', '-czf', str(asset_root / 'outputs.tar.gz'),
                    '-C', str(drive_project / 'outputs'),
                    'embeddings', 'colab_runs', 'answer_reward_cache.json'], check=True)
    credentials = json.loads(credential.read_text(encoding='utf-8'))
    metadata = {
        'title': 'CS 769 PPO Assets',
        'id': f"{credentials['username']}/cs-769-ppo-assets",
        'description': 'Private processed data, embeddings, selector checkpoints, and PPO reward cache.',
        'licenses': [{'name': 'other'}],
    }
    (asset_root / 'dataset-metadata.json').write_text(
        json.dumps(metadata, indent=2), encoding='utf-8'
    )
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle'], check=True)
    subprocess.run(['kaggle', 'datasets', 'create', '-p', str(asset_root), '-r', 'skip'], check=True)
    print('Published:', metadata['id'])
else:
    print('Asset publication skipped.')

## Completed experiment summary

This cell recomputes the table from saved prediction JSONL files. Its retained output records the completed official-dev experiment and remains visible when the notebook is opened.

In [13]:
from evaluation import aggregate_result_directory

result_dir = output_dir / 'results'
prediction_files = sorted(result_dir.glob('*.jsonl')) if result_dir.exists() else []
if prediction_files:
    scores = aggregate_result_directory(result_dir)
    ordered = sorted(
        scores['summary'].items(),
        key=lambda item: item[1]['answer_f1_mean'],
        reverse=True,
    )
    for method, metrics in ordered:
        print(
            f"{method:20s} Answer EM={metrics['answer_em_mean']:.4f} | "
            f"Answer F1={metrics['answer_f1_mean']:.4f} | "
            f"Evidence F1={metrics['evidence_f1_mean']:.4f}"
        )
    print('Saved metrics:', result_dir / 'metrics.json')
else:
    print('No prediction files yet; run final evaluation first.')

ppo_answer_only      Answer EM=0.1746 | Answer F1=0.2317 | Evidence F1=0.3250
ppo_combined         Answer EM=0.1716 | Answer F1=0.2271 | Evidence F1=0.3495
ppo_combined_step    Answer EM=0.1677 | Answer F1=0.2243 | Evidence F1=0.3541
ppo_evidence_only    Answer EM=0.1679 | Answer F1=0.2214 | Evidence F1=0.3529
supervised           Answer EM=0.1607 | Answer F1=0.2145 | Evidence F1=0.3685
similarity           Answer EM=0.0635 | Answer F1=0.1131 | Evidence F1=0.2616
direct               Answer EM=0.0332 | Answer F1=0.0826 | Evidence F1=nan
Saved metrics: outputs/results/metrics.json
